# 📈 02 — Holt-Winters (Triple Exponential Smoothing)
## Lissage exponentiel avec 4 configurations testées

Ce notebook importe les données préparées et teste Holt-Winters avec différentes configurations (additif/multiplicatif, damped/undamped).

## 1. 📦 Imports & Chargement

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✅ Imports OK")

In [ ]:
df = pd.read_csv("prepared_data.csv", index_col=0, parse_dates=True)
train = pd.read_csv("train_data.csv", index_col=0, parse_dates=True)
test = pd.read_csv("test_data.csv", index_col=0, parse_dates=True)

print(f"✅ Données chargées")
print(f"   Train : {len(train)} mois → Test : {len(test)} mois")

## 2. ⚙️ Test des 4 configurations Holt-Winters

In [ ]:
results_hw = {}
configs = [
    {"trend": "add", "seasonal": "add", "damped": True, "label": "add damped=True"},
    {"trend": "add", "seasonal": "multiplicative", "damped": True, "label": "multi damped=True"},
    {"trend": "add", "seasonal": "add", "damped": False, "label": "add damped=False"},
    {"trend": "add", "seasonal": "multiplicative", "damped": False, "label": "multi damped=False"},
]

print("⏳ Test des 4 configurations Holt-Winters...")
for cfg in configs:
    try:
        model = ExponentialSmoothing(train["revenue"], trend=cfg["trend"],
                                     seasonal=cfg["seasonal"], seasonal_periods=12,
                                     damped_trend=cfg["damped"])
        fitted = model.fit(optimized=True, use_brute=True)
        pred = fitted.forecast(12)
        pred.index = test.index
        mape_val = np.mean(np.abs((test["revenue"].values - pred.values) / test["revenue"].values)) * 100
        results_hw[cfg["label"]] = (pred, mape_val, fitted)
        print(f"   {cfg['label']:<25} → MAPE: {mape_val:.2f}%")
    except Exception as e:
        print(f"   {cfg['label']:<25} → Erreur: {e}")

best_key = min(results_hw, key=lambda k: results_hw[k][1])
hw_pred, best_mape, hw_fit = results_hw[best_key]
print(f"\n✅ Meilleur : {best_key} (MAPE: {best_mape:.2f}%)")

## 3. 📊 Évaluation & Graphiques

In [ ]:
# ── Métriques ─────────────────────────────────────────────
mae = mean_absolute_error(test["revenue"], hw_pred)
rmse = np.sqrt(mean_squared_error(test["revenue"], hw_pred))
r2 = r2_score(test["revenue"], hw_pred)

print(f"\n📊 Holt-Winters ({best_key})")
print(f"   MAE  : {mae:>12,.0f} €")
print(f"   RMSE : {rmse:>12,.0f} €")
print(f"   MAPE : {best_mape:>11.2f} %")
print(f"   R²   : {r2:>11.4f}")

# ── Graphique ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax = axes[0]
train["revenue"].plot(ax=ax, label="Train", color="#1565C0", alpha=0.7, linewidth=2)
test["revenue"].plot(ax=ax, label="Test réel", color="#2E7D32", linewidth=2.5)
hw_pred.plot(ax=ax, label=f"Holt-Winters ({best_key})", color="#F57C00", linestyle="--", linewidth=2.5)
hw_fit.fittedvalues.plot(ax=ax, label="Fitted (train)", color="#F57C00", alpha=0.35, linewidth=1)
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-30"), alpha=0.07, color="red", label="COVID")
ax.set_title(f"Holt-Winters ({best_key}) — Prévisions vs Réel", fontsize=13, fontweight="bold")
ax.set_ylabel("Revenue (€)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M€"))
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(hw_fit.level, label="Niveau", color="#1565C0", linewidth=2)
ax2_twin = ax2.twinx()
ax2_twin.plot(hw_fit.trend, label="Pente", color="#E65100", linewidth=2, linestyle="--")
ax2.set_title("Composantes : Niveau & Pente", fontsize=12)
ax2.set_ylabel("Niveau (€)", color="#1565C0")
ax2_twin.set_ylabel("Pente", color="#E65100"); ax2.grid(alpha=0.3)
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.suptitle(f"Holt-Winters | MAPE: {best_mape:.2f}% | R²: {r2:.4f}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("modele2_holtwinters.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n💾 Graphique → modele2_holtwinters.png")

# ── Sauvegarder ──────────────────────────────────────────
results_df = pd.DataFrame({"date": test.index, "reel": test["revenue"], "prediction": hw_pred})
results_df.to_csv("predictions_holtwinters.csv", index=False)
print("💾 Prédictions → predictions_holtwinters.csv")